In [ ]:
import json
import numpy as np
import librosa
import tensorflow as tf

from pathlib import Path


# ============================================================
# PROJECT DIRECTORY
# ============================================================

# Folder containing this Python script
PROJECT_DIR = Path(__file__).resolve().parent


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_PATH = PROJECT_DIR / "models" / "medium.keras"

AUDIO_PATH = PROJECT_DIR / "test_audio" / "marvin_test.wav"

NORMALIZATION_PATH = PROJECT_DIR / "models" / "normalization.json"


SAMPLE_RATE = 16000
AUDIO_LENGTH = 16000

N_MFCC = 13
N_FFT = 512
HOP_LENGTH = 320

CLASS_NAMES = [
    "marvin",
    "unknown",
    "background"
]


# ============================================================
# CHECK FILES
# ============================================================

if not MODEL_PATH.exists():

    raise FileNotFoundError(
        f"Model not found:\n{MODEL_PATH}"
    )

if not AUDIO_PATH.exists():

    raise FileNotFoundError(
        f"Audio file not found:\n{AUDIO_PATH}"
    )

if not NORMALIZATION_PATH.exists():

    raise FileNotFoundError(
        f"Normalization file not found:\n"
        f"{NORMALIZATION_PATH}\n\n"
        f"Run train_models.py first."
    )


# ============================================================
# LOAD MODEL
# ============================================================

print("=" * 60)
print("LOADING MODEL")
print("=" * 60)

print("Model:", MODEL_PATH)

model = tf.keras.models.load_model(
    MODEL_PATH
)

print(
    "Parameters:",
    model.count_params()
)


# ============================================================
# LOAD NORMALIZATION PARAMETERS
# ============================================================

print("\n")
print("=" * 60)
print("LOADING NORMALIZATION")
print("=" * 60)

print(
    "File:",
    NORMALIZATION_PATH
)

with open(
    NORMALIZATION_PATH,
    "r"
) as f:

    normalization = json.load(f)


TRAIN_MEAN = float(
    normalization["mean"]
)

TRAIN_STD = float(
    normalization["std"]
)


print(
    "Training mean:",
    TRAIN_MEAN
)

print(
    "Training std:",
    TRAIN_STD
)


# ============================================================
# LOAD REAL AUDIO
# ============================================================

print("\n")
print("=" * 60)
print("LOADING AUDIO")
print("=" * 60)

print("Audio:", AUDIO_PATH)


audio, sr = librosa.load(
    AUDIO_PATH,
    sr=SAMPLE_RATE,
    mono=True
)


print(
    "Sample rate:",
    sr
)

print(
    "Samples:",
    len(audio)
)

print(
    "Duration:",
    f"{len(audio) / SAMPLE_RATE:.3f}",
    "seconds"
)


# ============================================================
# MAKE EXACTLY 1 SECOND
# ============================================================

if len(audio) < AUDIO_LENGTH:

    print(
        "\nAudio shorter than 1 second."
    )

    print(
        "Padding with zeros..."
    )

    audio = np.pad(
        audio,
        (
            0,
            AUDIO_LENGTH - len(audio)
        )
    )

else:

    if len(audio) > AUDIO_LENGTH:

        print(
            "\nAudio longer than 1 second."
        )

        print(
            "Using first 1 second..."
        )

    audio = audio[:AUDIO_LENGTH]


print(
    "Final audio samples:",
    len(audio)
)


# ============================================================
# MFCC
# ============================================================

print("\n")
print("=" * 60)
print("EXTRACTING MFCC")
print("=" * 60)

mfcc = librosa.feature.mfcc(
    y=audio,
    sr=SAMPLE_RATE,
    n_mfcc=N_MFCC,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH
)


print(
    "Raw MFCC shape:",
    mfcc.shape
)


# ============================================================
# MAKE EXACTLY 49 FRAMES
# ============================================================

if mfcc.shape[1] < 49:

    print(
        "MFCC has fewer than 49 frames."
    )

    print(
        "Padding frames..."
    )

    mfcc = np.pad(
        mfcc,
        (
            (0, 0),
            (0, 49 - mfcc.shape[1])
        )
    )

else:

    if mfcc.shape[1] > 49:

        print(
            "MFCC has more than 49 frames."
        )

        print(
            "Using first 49 frames..."
        )

    mfcc = mfcc[:, :49]


print(
    "Fixed MFCC shape:",
    mfcc.shape
)


# ============================================================
# CHANGE SHAPE
# ============================================================

# (13, 49)
#     ↓
# (49, 13)
#     ↓
# (49, 13, 1)
#     ↓
# (1, 49, 13, 1)

mfcc = mfcc.T

mfcc = mfcc[..., np.newaxis]

mfcc = np.expand_dims(
    mfcc,
    axis=0
)


print(
    "Model input shape:",
    mfcc.shape
)


# ============================================================
# NORMALIZATION
# ============================================================

print("\n")
print("=" * 60)
print("NORMALIZING FEATURES")
print("=" * 60)

print(
    "Using saved training normalization values."
)

mfcc = (
    mfcc - TRAIN_MEAN
) / (
    TRAIN_STD + 1e-8
)


mfcc = mfcc.astype(
    np.float32
)


print(
    "Normalized dtype:",
    mfcc.dtype
)

print(
    "Normalized min:",
    f"{mfcc.min():.4f}"
)

print(
    "Normalized max:",
    f"{mfcc.max():.4f}"
)


# ============================================================
# RUN MODEL
# ============================================================

print("\n")
print("=" * 60)
print("RUNNING INFERENCE")
print("=" * 60)


probabilities = model.predict(
    mfcc,
    verbose=0
)


probabilities = probabilities[0]


# ============================================================
# RESULTS
# ============================================================

predicted_class = int(
    np.argmax(probabilities)
)

predicted_name = CLASS_NAMES[
    predicted_class
]

confidence = float(
    probabilities[predicted_class]
)


print("\nPrediction:")

print(
    "Class:",
    predicted_name
)

print(
    "Class ID:",
    predicted_class
)

print(
    "Confidence:",
    f"{confidence * 100:.2f}%"
)


# ============================================================
# ALL CLASS PROBABILITIES
# ============================================================

print("\n")
print("=" * 60)
print("CLASS PROBABILITIES")
print("=" * 60)

for i, name in enumerate(
    CLASS_NAMES
):

    print(
        f"{name:<12}: "
        f"{probabilities[i] * 100:.2f}%"
    )


# ============================================================
# FINAL RESULT
# ============================================================

print("\n")
print("=" * 60)
print("FINAL RESULT")
print("=" * 60)

if predicted_class == 0:

    print(
        ">>> WAKE WORD DETECTED <<<"
    )

else:

    print(
        ">>> NO WAKE WORD <<<"
    )

print("=" * 60)